In [1]:
import pandas as pd
from pathlib import Path
import holidays
import numpy as np

In [2]:
ROOT = Path.cwd().resolve().parent

DATA = ROOT / "Mayo26"
expediciones = DATA / "expediciones.xls"
frecuencias_fijas = DATA / "POT_V_VALPARAISOUN07_UN07_NORMAL_2026_A1_2.xlsx"

In [3]:
df_exp = pd.read_html(expediciones, flavor = 'lxml')[0]

In [15]:
df_exp.columns

Index(['Fecha', 'Inicio Expedicion', 'Fin Expedicion', 'Folio TS', 'ID Exp',
       'Bus', 'Chofer', 'Propietario', 'Variante', 'Servicio', 'Periodo',
       'Sentido', 'Estado', 'Causa', 'Tipo demanda', 'Frecuencia',
       'Vel.Promedio', 'Vel.Maxima', '1', '2', '3', '4', '5', '6', '7', '8',
       '9', '10', '11', '12', '13', '14', '15'],
      dtype='str')

In [12]:
df_exp.head()

,Fecha,Inicio Expedicion,Fin Expedicion,Folio TS,ID Exp,Bus,Chofer,Propietario,Variante,Servicio,...,6,7,8,9,10,11,12,13,14,15
0,2026-05-01,2026-05-01 06:56:36,2026-05-01 06:56:36,2605017620435,6886217,108/GWXF20,luis urquia lopez,ZASPIAK,762,702,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-05-01,2026-05-01 06:57:17,2026-05-01 06:57:17,2605017620435,6886218,108/GWXF20,luis urquia lopez,ZASPIAK,762,702,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-05-01,2026-05-01 06:59:52,2026-05-01 07:42:29,2605017310420,6886345,116/FRDH12,PRUEBA PRUEBA,GLADYS MONTOLLA .,731,701,...,07:18:30,07:20:00,07:23:26,07:25:40,07:31:51,07:36:12,07:37:48,07:38:46,07:42:29,NaN
3,2026-05-01,2026-05-01 07:00:34,2026-05-01 07:46:32,2605017150420,6886371,153/GWXF15,JUAN LUIS ZUÃIGA BRAVO,NaN,715,705,...,07:21:08,07:22:42,07:25:19,07:28:10,07:32:44,07:35:00,07:39:39,07:41:42,07:45:22,07:46:32
4,2026-05-01,2026-05-01 07:01:11,2026-05-01 07:45:46,2605017330430,6886287,219/DHYV15,Jorge Marcelo Maturana Maturana,TERMINAL PLACILLA,733,703,...,07:21:23,07:25:23,07:31:12,07:33:13,07:36:03,07:39:37,07:43:33,07:44:53,07:45:46,NaN


In [4]:
df_exp["Fecha"] = pd.to_datetime(df_exp["Fecha"])
anios = df_exp["Fecha"].dt.year.unique().tolist()
feriados_cl = holidays.Chile(years = anios)

In [5]:
es_feriado = df_exp['Fecha'].apply(lambda x: x in feriados_cl)
es_domingo = df_exp['Fecha'].dt.weekday == 6
es_sabado = df_exp['Fecha'].dt.weekday == 5

condiciones = [
    es_feriado | es_domingo,
    es_sabado
]

valores = ['DF', 'DS']

df_exp["tipo_dia"] = np.select(condiciones, valores, default = 'DL')

In [6]:
df_exp[df_exp["tipo_dia"] == 'DF'].iloc[:,0].unique().tolist() #verificacion de feriados, cumplio

[Timestamp('2026-05-01 00:00:00'),
 Timestamp('2026-05-03 00:00:00'),
 Timestamp('2026-05-10 00:00:00'),
 Timestamp('2026-05-17 00:00:00'),
 Timestamp('2026-05-21 00:00:00'),
 Timestamp('2026-05-24 00:00:00'),
 Timestamp('2026-05-31 00:00:00')]

In [9]:
df_frec = pd.read_excel(frecuencias_fijas, sheet_name='701-I', skiprows=10, header=[0, 1])
df_frec = df_frec.dropna(axis=1, how='all')
df_frec = df_frec[df_frec.iloc[:, 0].astype(str).str.strip().str.lower() != 'total']

In [ ]:
df_frec.columns = [
    'periodo', 'horario',
    'demanda_DL', 'frecuencia_DL', 
    'demanda_DS', 'frecuencia_DS', 
    'demanda_DF', 'frecuencia_DF'
]

df_dl = df_frec[['periodo', 'demanda_DL', 'frecuencia_DL']].copy()
df_dl['tipo_dia'] = 'DL'
df_dl.rename(columns={'demanda_DL': 'tipo_demanda', 'frecuencia_DL': 'frecuencia'}, inplace=True)

# DS
df_ds = df_frec[['periodo', 'demanda_DS', 'frecuencia_DS']].copy()
df_ds['tipo_dia'] = 'DS'
df_ds.rename(columns={'demanda_DS': 'tipo_demanda', 'frecuencia_DS': 'frecuencia'}, inplace=True)

# DF
df_df = df_frec[['periodo', 'demanda_DF', 'frecuencia_DF']].copy()
df_df['tipo_dia'] = 'DF'
df_df.rename(columns={'demanda_DF': 'tipo_demanda', 'frecuencia_DF': 'frecuencia'}, inplace=True)

# apilar dataFrames verticalmente
df_normalizado = pd.concat([df_dl, df_ds, df_df], ignore_index=True)
df_normalizado = df_normalizado[['periodo', 'tipo_dia', 'tipo_demanda', 'frecuencia']]

In [13]:
df_normalizado[(df_normalizado["tipo_dia"] == "DL") & (df_normalizado["tipo_demanda"].notna())] # ejemplo de filtro por tipo dia y valores no nulos (normativa no pide nada y son periodos de descanso como la madrugada)

,periodo,tipo_dia,tipo_demanda,frecuencia
6,6,DL,ALTA,8.0
7,7,DL,ALTA,8.0
8,8,DL,ALTA,7.0
9,9,DL,ALTA,7.0
10,10,DL,ALTA,6.0
11,11,DL,ALTA,6.0
12,12,DL,ALTA,6.0
13,13,DL,ALTA,6.0
14,14,DL,ALTA,6.0
15,15,DL,ALTA,6.0


In [14]:
excel_completo = pd.ExcelFile(frecuencias_fijas)
hojas_servicios = [hoja for hoja in excel_completo.sheet_names if '-' in hoja]

lista_dataframes = []

for hoja in hojas_servicios:
    # Extraemos el código del servicio y el sentido desde el nombre de la pestaña
    # Ej: '701-I' -> servicio='701', sentido='I'
    servicio, sentido = hoja.split('-')
    
    
    df_frec = pd.read_excel(excel_completo, sheet_name=hoja, skiprows=10, header=[0, 1])
    df_frec = df_frec.dropna(axis=1, how='all')
    df_frec = df_frec[df_frec.iloc[:, 0].astype(str).str.strip().str.lower() != 'total']
    
    if df_frec.empty:
        continue
        
    # Asignar las columnas aplanadas
    df_frec.columns = [
        'periodo', 'horario', 
        'demanda_DL', 'frecuencia_DL', 
        'demanda_DS', 'frecuencia_DS', 
        'demanda_DF', 'frecuencia_DF'
    ]
    
    # Separar y etiquetar por tipo de día
    # Laboral (DL)
    df_dl = df_frec[['periodo', 'demanda_DL', 'frecuencia_DL']].copy()
    df_dl['tipo_dia'] = 'DL'
    df_dl.rename(columns={'demanda_DL': 'tipo_demanda', 'frecuencia_DL': 'frecuencia'}, inplace=True)

    # Sábado (DS)
    df_ds = df_frec[['periodo', 'demanda_DS', 'frecuencia_DS']].copy()
    df_ds['tipo_dia'] = 'DS'
    df_ds.rename(columns={'demanda_DS': 'tipo_demanda', 'frecuencia_DS': 'frecuencia'}, inplace=True)

    # Domingo / Festivo (DF)
    df_df = df_frec[['periodo', 'demanda_DF', 'frecuencia_DF']].copy()
    df_df['tipo_dia'] = 'DF'
    df_df.rename(columns={'demanda_DF': 'tipo_demanda', 'frecuencia_DF': 'frecuencia'}, inplace=True)
    
    # Apilar los días de la pestaña actual
    df_hoja_normalizado = pd.concat([df_dl, df_ds, df_df], ignore_index=True)
    
    # Agregar las columnas de identificación del servicio
    df_hoja_normalizado['servicio'] = servicio
    df_hoja_normalizado['sentido'] = sentido
    
    lista_dataframes.append(df_hoja_normalizado)

In [16]:
df_master_frecuencias = pd.concat(lista_dataframes, ignore_index=True)
df_master_frecuencias = df_master_frecuencias[[
    'servicio', 'sentido', 'periodo', 'tipo_dia', 'tipo_demanda', 'frecuencia'
]]

In [19]:
df_master_frecuencias

,servicio,sentido,periodo,tipo_dia,tipo_demanda,frecuencia
0,701,I,0,DL,NaN,NaN
1,701,I,1,DL,NaN,NaN
2,701,I,2,DL,NaN,NaN
3,701,I,3,DL,NaN,NaN
4,701,I,4,DL,NaN,NaN
...,...,...,...,...,...,...
859,706,R,19,DF,MEDIA,3.0
860,706,R,20,DF,MEDIA,3.0
861,706,R,21,DF,NaN,NaN
862,706,R,22,DF,NaN,NaN


In [21]:
df_master_frecuencias[df_master_frecuencias["tipo_demanda"].notna()]

,servicio,sentido,periodo,tipo_dia,tipo_demanda,frecuencia
6,701,I,6,DL,ALTA,8.0
7,701,I,7,DL,ALTA,8.0
8,701,I,8,DL,ALTA,7.0
9,701,I,9,DL,ALTA,7.0
10,701,I,10,DL,ALTA,6.0
...,...,...,...,...,...,...
856,706,R,16,DF,MEDIA,3.0
857,706,R,17,DF,MEDIA,3.0
858,706,R,18,DF,MEDIA,3.0
859,706,R,19,DF,MEDIA,3.0


se tiene las frecuencias y las expediciones, ahora deberiamos poder calcular ICF

In [22]:
# filtramos frecuencias
df_master_frecuencias["periodo"] = pd.to_numeric(df_master_frecuencias["periodo"], errors='coerce').astype('Int64')
df_master_frecuencias[(df_master_frecuencias["servicio"] == "701") & (df_master_frecuencias["sentido"] == "I") & (df_master_frecuencias["tipo_dia"] == "DL") & (df_master_frecuencias["frecuencia"].notna())]

,servicio,sentido,periodo,tipo_dia,tipo_demanda,frecuencia
6,701,I,6,DL,ALTA,8.0
7,701,I,7,DL,ALTA,8.0
8,701,I,8,DL,ALTA,7.0
9,701,I,9,DL,ALTA,7.0
10,701,I,10,DL,ALTA,6.0
11,701,I,11,DL,ALTA,6.0
12,701,I,12,DL,ALTA,6.0
13,701,I,13,DL,ALTA,6.0
14,701,I,14,DL,ALTA,6.0
15,701,I,15,DL,ALTA,6.0


In [ ]:
df_exp.columns 

Index(['Fecha', 'Inicio Expedicion', 'Fin Expedicion', 'Folio TS', 'ID Exp',
       'Bus', 'Chofer', 'Propietario', 'Variante', 'Servicio', 'Periodo',
       'Sentido', 'Estado', 'Causa', 'Tipo demanda', 'Frecuencia',
       'Vel.Promedio', 'Vel.Maxima', '1', '2', '3', '4', '5', '6', '7', '8',
       '9', '10', '11', '12', '13', '14', '15', 'tipo_dia'],
      dtype='str')

* estado -> si es valido o no la expedicion
* periodo -> el periodo al que pertenece la expedicion
* sentido -> Ida / Reg
* tipo demanda -> Baja Media Alta
* tipo_dia -> DL DS DF

In [23]:
df_exp["sentido_str"] = df_exp["Sentido"].astype(str).str.strip().str.upper().str[0]
df_exp["servicio_str"] = df_exp["Servicio"].astype(str).str.strip()
df_exp["periodo_num"] = pd.to_numeric(df_exp["Periodo"], errors='coerce').astype('Int64')

filtro_valida = df_exp["Estado"].astype(str).str.strip() == "Valida"
df_exp_validas = df_exp[filtro_valida].copy()

df_conteo = df_exp_validas.groupby(
    ['Fecha', 'servicio_str', 'sentido_str', 'tipo_dia', 'periodo_num']
).size().reset_index(name="expediciones_observadas")

df_conteo.rename(columns={
    'servicio_str': 'servicio', 
    'sentido_str': 'sentido', 
    'periodo_num': 'periodo' # Lo volvemos a llamar periodo para el cruce
}, inplace=True)

In [47]:
df_conteo[(df_conteo["servicio"] == "701") & (df_conteo["sentido"] == "I") & (df_conteo["periodo"] == 20)]

,Fecha,servicio,sentido,tipo_dia,periodo,expediciones_observadas
14,2026-05-01,701,I,DF,20,1
172,2026-05-02,701,I,DS,20,3
352,2026-05-03,701,I,DF,20,3
521,2026-05-04,701,I,DL,20,5
702,2026-05-05,701,I,DL,20,3
882,2026-05-06,701,I,DL,20,2
1061,2026-05-07,701,I,DL,20,3
1239,2026-05-08,701,I,DL,20,4
1416,2026-05-09,701,I,DS,20,3
1595,2026-05-10,701,I,DF,20,2


In [50]:
calendario = df_exp[["Fecha","tipo_dia"]].drop_duplicates()
df_base_exigida = pd.merge(calendario, df_master_frecuencias, on = "tipo_dia", how = "left")
df_base_exigida = df_base_exigida.dropna(subset=["frecuencia"])

df_cruce = pd.merge(
    df_base_exigida,
    df_conteo,
    on = ["Fecha","servicio","sentido","tipo_dia","periodo"],
)

df_cruce["expediciones_observadas"] = df_cruce["expediciones_observadas"].fillna(0)

<center>

|MES    |$\psi$|
|--     | -----  |
|mes<=24|0.9     |
|mes>=25|0.95    |

</center>

In [51]:
def calcular_factor_pago(icf_promedio, mes_operacion):
    """
    Aplica las reglas del Factor de Pago según el promedio de la demanda
    y el mes de operación (define la exigencia psi).
    """
    psi = 0.90 if mes_operacion <= 24 else 0.95
    if icf_promedio < 0.50:
        return 0.50
    elif icf_promedio > psi:
        return 1.00
    else:
        # Si está entre 0.50 y psi, el factor es igual al promedio
        return icf_promedio

In [52]:
df_cruce["tope_observadas"] = df_cruce[["expediciones_observadas", "frecuencia"]].min(axis = 1)
df_cruce["icf_base"] = (df_cruce["tope_observadas"] / df_cruce["frecuencia"]).round(2)

In [54]:
resumen_unidad = df_cruce.groupby("tipo_demanda")["icf_base"].mean().reset_index()
resumen_unidad.rename(columns = {"icf_base": "promedio_base"}, inplace = True)

resumen_unidad["factor_pago_demanda"] = resumen_unidad["promedio_base"].apply(
    lambda x: calcular_factor_pago(x, mes_operacion=25)
)

In [57]:
df_cruce

,Fecha,tipo_dia,servicio,sentido,periodo,tipo_demanda,frecuencia,expediciones_observadas,tope_observadas,icf_base
0,2026-05-01,DF,701,I,7,MEDIA,5.0,2,2.0,0.40
1,2026-05-01,DF,701,I,8,MEDIA,5.0,2,2.0,0.40
2,2026-05-01,DF,701,I,9,MEDIA,5.0,2,2.0,0.40
3,2026-05-01,DF,701,I,10,MEDIA,5.0,2,2.0,0.40
4,2026-05-01,DF,701,I,11,MEDIA,5.0,2,2.0,0.40
...,...,...,...,...,...,...,...,...,...,...
5371,2026-05-31,DF,706,R,15,MEDIA,3.0,2,2.0,0.67
5372,2026-05-31,DF,706,R,16,MEDIA,3.0,2,2.0,0.67
5373,2026-05-31,DF,706,R,17,MEDIA,3.0,3,3.0,1.00
5374,2026-05-31,DF,706,R,18,MEDIA,3.0,1,1.0,0.33


In [56]:
resumen_unidad

,tipo_demanda,promedio_base,factor_pago_demanda
0,ALTA,0.701996,0.701996
1,BAJA,0.750627,0.750627
2,MEDIA,0.669420,0.669420


In [55]:
icf_unico_mensual = resumen_unidad["factor_pago_demanda"].mean()
icf_unico_mensual

np.float64(0.7073475319698473)

In [ ]:
# df_cruce = pd.merge(
#     df_conteo,
#     df_master_frecuencias,
#     on = ["servicio","sentido","periodo","tipo_dia"],
#     how = "left"
# )
# #quitamos los registros en que la normativa no define frecuencia de buses
# df_cruce = df_cruce.dropna(subset=["frecuencia"])
# df_cruce.head()

,Fecha,servicio,sentido,tipo_dia,periodo,expediciones_observadas,tipo_demanda,frecuencia
1,2026-05-01,701,I,DF,7,2,MEDIA,5.0
2,2026-05-01,701,I,DF,8,2,MEDIA,5.0
3,2026-05-01,701,I,DF,9,2,MEDIA,5.0
4,2026-05-01,701,I,DF,10,2,MEDIA,5.0
5,2026-05-01,701,I,DF,11,2,MEDIA,5.0


In [58]:
# #calculo icf
# #no podemos tener mas expediciones que las exigidas (no se consideran)
# df_cruce["tope_observadas"] = df_cruce[["expediciones_observadas", "frecuencia"]].min(axis = 1)
# #porcentaje cumplimiento
# df_cruce["icf_base"] = (df_cruce["tope_observadas"] / df_cruce["frecuencia"]).round(2)
#agrupamos por tipo de demanda
resumen_mensual = df_cruce.groupby(["servicio","sentido","tipo_demanda"])['icf_base'].mean().reset_index()
resumen_mensual.rename(columns = {"icf_base":"promedio_icf_demanda"}, inplace = True)
resumen_mensual["promedio_icf_demanda"] = resumen_mensual["promedio_icf_demanda"].round(2)

In [59]:
resumen_mensual

,servicio,sentido,tipo_demanda,promedio_icf_demanda
0,701,I,ALTA,0.89
1,701,I,BAJA,0.93
2,701,I,MEDIA,0.78
3,701,R,ALTA,0.89
4,701,R,BAJA,0.78
5,701,R,MEDIA,0.80
6,702,I,ALTA,0.71
7,702,I,BAJA,0.79
8,702,I,MEDIA,0.74
9,702,R,ALTA,0.64


In [62]:
icf_unico_mensual = resumen_unidad["factor_pago_demanda"].mean()

print("=== REPORTE UNIDAD DE NEGOCIO ===")
display(resumen_unidad)
print(f"-> ICF ÚNICO DE LA UNIDAD: {icf_unico_mensual:.2f}\n")

=== REPORTE UNIDAD DE NEGOCIO ===


,tipo_demanda,promedio_base,factor_pago_demanda
0,ALTA,0.701996,0.701996
1,BAJA,0.750627,0.750627
2,MEDIA,0.669420,0.669420


-> ICF ÚNICO DE LA UNIDAD: 0.71



In [63]:
resumen_mensual["factor_pago"] = resumen_mensual["promedio_icf_demanda"].apply(
    lambda x: calcular_factor_pago(x, mes_operacion=25) # Usamos 25 tal como pusiste arriba
)

In [114]:
resumen_mensual

,servicio,sentido,tipo_demanda,promedio_icf_demanda,factor_pago
0,701,I,ALTA,0.89,0.89
1,701,I,BAJA,0.93,0.93
2,701,I,MEDIA,0.78,0.78
3,701,R,ALTA,0.89,0.89
4,701,R,BAJA,0.78,0.78
5,701,R,MEDIA,0.80,0.80
6,702,I,ALTA,0.71,0.71
7,702,I,BAJA,0.79,0.79
8,702,I,MEDIA,0.74,0.74
9,702,R,ALTA,0.64,0.64


In [64]:
icf_por_servicio = resumen_mensual.groupby("servicio")["factor_pago"].mean().reset_index()

# C. Renombramos y redondeamos a 2 decimales para que quede idéntico al reporte de tu experto
icf_por_servicio.rename(columns={"factor_pago": "ICF"}, inplace=True)
icf_por_servicio["ICF"] = icf_por_servicio["ICF"].round(2)

print("=== REPORTE POR SERVICIO (Ranking) ===")
display(icf_por_servicio)

=== REPORTE POR SERVICIO (Ranking) ===


,servicio,ICF
0,701,0.85
1,702,0.71
2,703,0.63
3,704,0.72
4,705,0.70
5,706,0.65


In [112]:
icf_mensual_final = resumen_mensual.groupby(['servicio', 'sentido'])['factor_pago'].mean().reset_index()
icf_mensual_final.rename(columns={'factor_pago': 'ICF_Mensual'}, inplace=True)
icf_mensual_final['ICF_Mensual'] = icf_mensual_final['ICF_Mensual'].round(2)

In [115]:
icf_mensual_final

,servicio,sentido,ICF_Mensual
0,701,I,0.87
1,701,R,0.82
2,702,I,0.75
3,702,R,0.67
4,703,I,0.58
5,703,R,0.68
6,704,I,0.72
7,704,R,0.72
8,705,I,0.68
9,705,R,0.73


In [117]:
resumen_UN = df_cruce.groupby('tipo_demanda')['icf_base'].mean().reset_index()
resumen_UN.rename(columns={'icf_base': 'promedio_base'}, inplace=True)

In [118]:
resumen_UN

,tipo_demanda,promedio_base
0,ALTA,0.701996
1,BAJA,0.750627
2,MEDIA,0.669420


In [120]:
resumen_UN['factor_pago_demanda'] = resumen_UN['promedio_base'].apply(
    lambda x: calcular_factor_pago(x, mes_operacion=MES_DE_OPERACION)
)

In [121]:
resumen_UN

,tipo_demanda,promedio_base,factor_pago_demanda
0,ALTA,0.701996,0.701996
1,BAJA,0.750627,0.750627
2,MEDIA,0.669420,0.669420


In [125]:
icf_unico_mensual = resumen_UN['factor_pago_demanda'].mean()
print("Desglose por Tipo de Demanda de toda la flota:")
display(resumen_UN)
print(f"\n---> EL ICF ÚNICO DE LA UNIDAD (FACTOR DE PAGO) ES: {icf_unico_mensual:.2f} <---")

Desglose por Tipo de Demanda de toda la flota:


,tipo_demanda,promedio_base,factor_pago_demanda
0,ALTA,0.701996,0.701996
1,BAJA,0.750627,0.750627
2,MEDIA,0.669420,0.669420



---> EL ICF ÚNICO DE LA UNIDAD (FACTOR DE PAGO) ES: 0.71 <---
